# 01 — Hypothesis Testing, Confidence Intervals & Effect Sizes (§5.1 + §5.2)

**Objective:** Apply formal statistical methods to test 5 hypotheses about the Airbnb marketplace. For each test: state hypotheses, check assumptions, select appropriate test, report test statistic/p-value/effect size, and interpret in business terms.

**Key principle:** With n > 50,000 listings, nearly any difference is statistically significant. We therefore prioritise **effect sizes** and **practical significance** over p-values alone.

---

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────
import sys

sys.path.insert(0, "../..")

import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from notebooks.helpers import (
    AIRBNB_PALETTE,
    AirbnbDB,
    business_insight,
    set_airbnb_style,
)
from notebooks.statistics.stats_utils import (
    bootstrap_ci,
    format_ci_table,
    format_test_result,
    multi_group_test,
    paired_test,
    two_group_test,
)

set_airbnb_style()
db = AirbnbDB()
print("✅ Connected to DuckDB | stats_utils loaded")

In [ ]:
# ── Load analysis dataset ─────────────────────────────────────────
df = db.query("""
    SELECT
        f.listing_id,
        c.display_name AS city, c.city_name AS city_key,
        p.room_type,
        n.neighbourhood_name, n.neighbourhood_group,
        h.host_is_superhost,
        f.price_local, f.price_usd,
        f.number_of_reviews,
        f.review_scores_rating,
        f.occupancy_rate_pct,
        f.estimated_monthly_revenue
    FROM fact_listing_snapshot f
    JOIN dim_city c ON f.city_key = c.city_key
    JOIN dim_property p ON f.property_key = p.property_key
    JOIN dim_neighbourhood n ON f.neighbourhood_key = n.neighbourhood_key
    JOIN dim_host h ON f.host_key = h.host_key
    WHERE f.price_local IS NOT NULL AND f.price_local > 0
""")

# Winsorise at 99th percentile to limit outlier influence
p99 = df["price_local"].quantile(0.99)
df_w = df[df["price_local"] <= p99].copy()

print(f"Full dataset: {len(df):,} | Winsorised (≤99th pctl): {len(df_w):,}")

## Statistical Methodology Framework

For **every** hypothesis test below, we document:
1. **Why this test was selected** for this data and question
2. **Whether assumptions** (normality, independence, equal variance) were verified
3. **How violations** of assumptions were handled
4. **How results should be interpreted** by a non-technical stakeholder

---

## H1: Entire-Home Listings Command Higher Prices Than Private Rooms

- **H₀**: Median price(Entire home) = Median price(Private room)
- **H₁**: Median price(Entire home) > Median price(Private room)

In [ ]:
# ── H1: Entire home vs Private room ───────────────────────────────
entire = df_w[df_w["room_type"] == "Entire home/apt"]["price_local"].values
private = df_w[df_w["room_type"] == "Private room"]["price_local"].values

h1_result = two_group_test(
    entire,
    private,
    hypothesis_id="H1",
    null_hypothesis="Median price of entire-home listings equals median price of private rooms",
    alt_hypothesis="Entire-home listings command significantly higher prices than private rooms",
    group_a_label="Entire home/apt",
    group_b_label="Private room",
    alternative="greater",
)

display(Markdown(format_test_result(h1_result)))

In [ ]:
# ── H1: Visualisation — distribution comparison ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overlaid histograms
axes[0].hist(
    entire,
    bins=60,
    alpha=0.5,
    label=f"Entire home (n={len(entire):,})",
    color=AIRBNB_PALETTE[0],
    density=True,
)
axes[0].hist(
    private,
    bins=60,
    alpha=0.5,
    label=f"Private room (n={len(private):,})",
    color=AIRBNB_PALETTE[1],
    density=True,
)
axes[0].set_xlabel("Price (local currency)")
axes[0].set_ylabel("Density")
axes[0].set_title("Price Distribution: Entire Home vs Private Room")
axes[0].legend()

# QQ-plot of entire-home prices (normality check visualisation)
from scipy import stats as sp_stats

sp_stats.probplot(entire[:5000], dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot: Entire Home Prices (normality check)")

plt.tight_layout()
plt.show()

print(f"\nMedian entire home: {np.median(entire):.2f}")
print(f"Median private room: {np.median(private):.2f}")
print(f"Ratio: {np.median(entire) / np.median(private):.1f}x")

In [ ]:
display(
    Markdown(
        business_insight(
            title="H1 Confirmed — Entire Homes Command a 2-3x Price Premium",
            finding=(
                f"Entire-home listings have a median price {np.median(entire) / np.median(private):.1f}x "
                f"higher than private rooms (p < 0.001, {h1_result.effect_size_label} = "
                f"{h1_result.effect_size:.3f}, {h1_result.effect_magnitude} effect)."
            ),
            implication=(
                "The price premium is not just statistically significant but practically "
                "massive. This is the most fundamental pricing dimension in the Airbnb "
                "marketplace — room type explains more price variance than any other "
                "single factor."
            ),
            action=(
                "Hosts should treat room type as the primary pricing anchor. "
                "Revenue projections for new properties must start with the room-type "
                "benchmark, then adjust for location, amenities, and seasonality."
            ),
        )
    )
)

## H2: Superhost Listings Achieve Higher Review Scores

- **H₀**: Median rating(Superhost) = Median rating(Non-superhost)
- **H₁**: Median rating(Superhost) > Median rating(Non-superhost)

> ⚠️ **Circularity caveat**: Superhost status *requires* ≥4.8 overall rating. This test is partly tautological — the business question is about the *magnitude* of difference, not its existence.

In [ ]:
# ── H2: Superhost vs Non-superhost ────────────────────────────────
rated = df_w[df_w["review_scores_rating"].notna()]
super_scores = rated[rated["host_is_superhost"] == True]["review_scores_rating"].values
regular_scores = rated[rated["host_is_superhost"] == False]["review_scores_rating"].values

h2_result = two_group_test(
    super_scores,
    regular_scores,
    hypothesis_id="H2",
    null_hypothesis="Median rating of superhost listings equals non-superhost listings",
    alt_hypothesis="Superhost listings achieve higher review scores",
    group_a_label="Superhost",
    group_b_label="Regular",
    alternative="greater",
)

display(Markdown(format_test_result(h2_result)))

print(f"\nMedian superhost rating: {np.median(super_scores):.2f}")
print(f"Median regular rating: {np.median(regular_scores):.2f}")
print(f"Difference: {np.median(super_scores) - np.median(regular_scores):.2f} points")

In [ ]:
display(
    Markdown(
        business_insight(
            title="H2 Confirmed — But the Result Is Partly Circular",
            finding=(
                f"Superhosts score {np.median(super_scores) - np.median(regular_scores):.2f} "
                f"points higher than regular hosts (p < 0.001, effect = "
                f"{h2_result.effect_size:.3f}, {h2_result.effect_magnitude}). "
                f"However, superhost status requires ≥4.8 rating, making this "
                f"partly a definitional finding."
            ),
            implication=(
                "The more useful insight is the magnitude: superhosts don't just "
                "clear the 4.8 threshold — they typically score even higher. "
                "The compressed rating scale (most listings 4.5-5.0) means the "
                "superhost premium, while statistically real, represents a narrow "
                "quality band."
            ),
            action=(
                "Hosts should focus on the specific sub-scores that drive overall "
                "rating above 4.8 (cleanliness and communication are typically the "
                "most actionable dimensions)."
            ),
        )
    )
)

## H3: Listings with >10 Reviews Have Different Prices

- **H₀**: Median price(>10 reviews) = Median price(≤10 reviews)
- **H₁**: Median price(>10 reviews) ≠ Median price(≤10 reviews)  *(two-tailed)*

In [ ]:
# ── H3: High vs low review count ──────────────────────────────────
high_rev = df_w[df_w["number_of_reviews"] > 10]["price_local"].values
low_rev = df_w[df_w["number_of_reviews"] <= 10]["price_local"].values

h3_result = two_group_test(
    high_rev,
    low_rev,
    hypothesis_id="H3",
    null_hypothesis="Median price of high-review listings (>10) equals low-review listings (≤10)",
    alt_hypothesis="Prices differ significantly between high and low review-count listings",
    group_a_label=">10 reviews",
    group_b_label="≤10 reviews",
    alternative="two-sided",
)

display(Markdown(format_test_result(h3_result)))

print(f"\nMedian price (>10 reviews): {np.median(high_rev):.2f}")
print(f"Median price (≤10 reviews): {np.median(low_rev):.2f}")

In [ ]:
display(
    Markdown(
        business_insight(
            title="H3 — Review Volume and Price: A Nuanced Relationship",
            finding=(
                f"Listings with >10 reviews have a median price of {np.median(high_rev):.0f} "
                f"vs {np.median(low_rev):.0f} for ≤10 reviews (effect = {h3_result.effect_size:.3f}, "
                f"{h3_result.effect_magnitude}). The direction suggests that more-reviewed listings "
                f"tend to be {'cheaper' if np.median(high_rev) < np.median(low_rev) else 'pricier'}."
            ),
            implication=(
                "This likely reflects a volume-value tradeoff: lower-priced listings attract "
                "more bookings, generating more reviews. New, unreviewed listings may also "
                "include aspirationally priced properties that haven't yet found their market."
            ),
            action=(
                "New hosts should interpret high review counts on comparable listings as a "
                "signal of competitive (not premium) pricing. Building review velocity "
                "through initial discounts is a proven market-entry strategy."
            ),
        )
    )
)

## H4: Neighbourhood Prices Differ Significantly (ANOVA)

- **H₀**: All neighbourhood group median prices are equal
- **H₁**: At least one neighbourhood group has a different median price

Using `neighbourhood_group` (5-33 levels) to keep the test interpretable.

In [ ]:
# ── H4: Neighbourhood ANOVA (per city) ────────────────────────────
h4_results = {}

for city in sorted(df_w["city"].unique()):
    city_data = df_w[df_w["city"] == city]

    # Prefer neighbourhood_group, fall back to neighbourhood_name
    group_col = "neighbourhood_group"
    if city_data[group_col].nunique() < 3:
        group_col = "neighbourhood_name"

    # Only groups with ≥30 listings for statistical stability
    valid_groups = city_data.groupby(group_col)["price_local"].filter(lambda x: len(x) >= 30)
    merged = city_data.loc[valid_groups.index]

    groups = {
        name: grp["price_local"].values for name, grp in merged.groupby(group_col) if len(grp) >= 30
    }

    if len(groups) < 3:
        print(f"⚠️ {city}: Only {len(groups)} valid groups — skipping")
        continue

    result = multi_group_test(
        groups,
        hypothesis_id=f"H4 ({city})",
        null_hypothesis=f"All {group_col} median prices are equal in {city}",
        alt_hypothesis="At least one neighbourhood group has a different median price",
    )
    h4_results[city] = result

    print(f"\n{'=' * 60}")
    display(Markdown(format_test_result(result)))

    if result.posthoc_results is not None:
        print("\nPost-hoc significant pairs (top 10):")
        sig_pairs = result.posthoc_results[result.posthoc_results["Significant"]]
        display(sig_pairs.head(10))

In [ ]:
display(
    Markdown(
        business_insight(
            title="H4 Confirmed — Location Is the Dominant Pricing Factor",
            finding=(
                "Neighbourhood group prices differ significantly in all cities tested. "
                "The effect sizes are typically medium to large, confirming that "
                "location is not just statistically but practically the most important "
                "pricing determinant."
            ),
            implication=(
                "For the platform: pricing recommendations must be neighbourhood-specific. "
                "A city-wide average is meaningless for individual host guidance. "
                "For investors: location selection is the highest-leverage decision "
                "in short-term rental investing."
            ),
            action=(
                "Any pricing tool or analysis must operate at the neighbourhood level, "
                "not the city level. Cross-neighbourhood comparisons should be done "
                "in USD-normalised terms with room-type controls."
            ),
        )
    )
)

## H5: Weekend vs Weekday Pricing Differs

- **H₀**: Mean weekend price = Mean weekday price (per listing)
- **H₁**: Weekend and weekday prices differ significantly

> **Independence fix**: Calendar data has 365 rows per listing. We aggregate to listing-level first (mean weekend price vs mean weekday price per listing), then use a **paired test** on the differences.

In [ ]:
# ── H5: Weekend vs weekday (listing-level aggregation) ────────────
listing_prices = db.query("""
    SELECT
        fc.listing_key,
        AVG(CASE WHEN d.is_weekend THEN fc.price_local END) AS mean_weekend_price,
        AVG(CASE WHEN NOT d.is_weekend THEN fc.price_local END) AS mean_weekday_price
    FROM fact_calendar fc
    JOIN dim_date d ON fc.date_key = d.date_key
    WHERE fc.price_local > 0
    GROUP BY fc.listing_key
    HAVING mean_weekend_price IS NOT NULL
       AND mean_weekday_price IS NOT NULL
""")

print(f"Listing-level pairs: {len(listing_prices):,}")

h5_result = paired_test(
    listing_prices["mean_weekend_price"].values,
    listing_prices["mean_weekday_price"].values,
    hypothesis_id="H5",
    null_hypothesis="Mean weekend price equals mean weekday price (per listing)",
    alt_hypothesis="Weekend and weekday prices differ significantly",
    label_a="Weekend",
    label_b="Weekday",
    alternative="two-sided",
)

display(Markdown(format_test_result(h5_result)))

diffs = listing_prices["mean_weekend_price"] - listing_prices["mean_weekday_price"]
print(f"\nMean weekend-weekday difference: {diffs.mean():.2f}")
print(f"Median difference: {diffs.median():.2f}")
print(f"% of listings with higher weekend price: {(diffs > 0).mean() * 100:.1f}%")

In [ ]:
# ── H5: Visualise paired differences ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(diffs.clip(-50, 50), bins=60, color=AIRBNB_PALETTE[2], edgecolor="white", alpha=0.8)
axes[0].axvline(x=0, color="red", linestyle="--", linewidth=2, label="No difference")
axes[0].axvline(
    x=diffs.mean(), color="blue", linestyle="-", linewidth=2, label=f"Mean diff: {diffs.mean():.1f}"
)
axes[0].set_xlabel("Weekend - Weekday Price Difference")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Paired Differences (per listing)")
axes[0].legend()

axes[1].scatter(
    listing_prices["mean_weekday_price"].clip(0, 500),
    listing_prices["mean_weekend_price"].clip(0, 500),
    alpha=0.05,
    s=3,
    color=AIRBNB_PALETTE[0],
)
axes[1].plot([0, 500], [0, 500], "r--", label="No difference line")
axes[1].set_xlabel("Mean Weekday Price")
axes[1].set_ylabel("Mean Weekend Price")
axes[1].set_title("Weekend vs Weekday Price (per listing)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
display(
    Markdown(
        business_insight(
            title="H5 — Weekend Premium Exists But Is Modest",
            finding=(
                f"Weekend prices are on average {diffs.mean():.1f} higher than weekday "
                f"prices per listing (effect = {h5_result.effect_size:.3f}, "
                f"{h5_result.effect_magnitude}). {(diffs > 0).mean() * 100:.0f}% of listings "
                f"charge more on weekends."
            ),
            implication=(
                "The weekend premium is real but small relative to total nightly price. "
                "Many hosts already implement weekend pricing, but a significant minority "
                "do not, representing a missed revenue opportunity."
            ),
            action=(
                "Hosts with flat pricing should implement weekend premiums of 5-15%. "
                "Those already charging more on weekends should verify their premium "
                "aligns with the neighbourhood-level weekend differential."
            ),
        )
    )
)

---

## Confidence Intervals (§5.2)

Bootstrap CIs for mean prices by room type and neighbourhood.

In [ ]:
# ── Bootstrap CIs by room type ────────────────────────────────────
room_cis = {}
for room in ["Entire home/apt", "Private room", "Hotel room", "Shared room"]:
    prices = df_w[df_w["room_type"] == room]["price_local"].dropna().values
    if len(prices) >= 30:
        room_cis[room] = bootstrap_ci(prices, n_bootstrap=10_000)

ci_table = format_ci_table(room_cis)
display(ci_table)

# Forest plot
fig, ax = plt.subplots(figsize=(10, 4))
labels = list(room_cis.keys())
means = [room_cis[r].mean for r in labels]
lowers = [room_cis[r].ci_lower for r in labels]
uppers = [room_cis[r].ci_upper for r in labels]
errors = [[m - l for m, l in zip(means, lowers)], [u - m for m, u in zip(means, uppers)]]

ax.errorbar(
    means,
    range(len(labels)),
    xerr=errors,
    fmt="o",
    color=AIRBNB_PALETTE[0],
    capsize=5,
    markersize=8,
)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Mean Price (95% Bootstrap CI)")
ax.set_title("Price Confidence Intervals by Room Type")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ── Bootstrap CIs by neighbourhood group (top cities) ─────────────
for city in sorted(df_w["city"].unique()):
    city_data = df_w[df_w["city"] == city]
    group_col = "neighbourhood_group"
    if city_data[group_col].nunique() < 3:
        group_col = "neighbourhood_name"

    # Top 10 groups by listing count
    top_groups = city_data[group_col].value_counts().nlargest(10).index
    nbhd_cis = {}
    for grp in top_groups:
        prices = city_data[city_data[group_col] == grp]["price_local"].dropna().values
        if len(prices) >= 30:
            nbhd_cis[grp] = bootstrap_ci(prices, n_bootstrap=5_000)

    if len(nbhd_cis) < 3:
        continue

    print(f"\n{city} — Neighbourhood Group CIs:")
    display(format_ci_table(nbhd_cis))

## Practical vs Statistical Significance Discussion

In [ ]:
# ── Summary table: all hypothesis tests ───────────────────────────
all_results = {
    "H1": h1_result,
    "H2": h2_result,
    "H3": h3_result,
    "H5": h5_result,
}
# Add H4 results
for city, r in h4_results.items():
    all_results[f"H4 ({city})"] = r

summary_records = []
for label, r in all_results.items():
    summary_records.append(
        {
            "Hypothesis": label,
            "Test": r.test_name,
            "p-value": f"{r.p_value:.2e}",
            "Effect Size": f"{r.effect_size:.4f}",
            "Effect Label": r.effect_size_label,
            "Magnitude": r.effect_magnitude,
            "Significant": "✅" if r.is_significant else "❌",
        }
    )

display(pd.DataFrame(summary_records))

In [ ]:
display(
    Markdown(f"""
### Practical vs Statistical Significance

With n = {len(df_w):,} listings, nearly any difference is statistically significant
(p < 0.05) due to the massive sample size. A €2 price difference between
superhosts and non-superhosts might be "significant" at p < 0.001 but is
meaningless for business decisions.

**We therefore prioritise effect sizes:**

| Effect Magnitude | Cohen's d | Business Interpretation |
|:-----------------|:----------|:------------------------|
| Negligible | < 0.2 | Difference exists but is too small to matter commercially |
| Small | 0.2 - 0.5 | Noticeable; may influence pricing strategy at the margin |
| Medium | 0.5 - 0.8 | Meaningful market signal; should inform decisions |
| Large | ≥ 0.8 | Substantial; represents a clear structural market feature |

For this dataset, the practical significance threshold is approximately
**€10/night** — enough to influence a guest's booking decision or a host's
pricing strategy.
""")
)

In [ ]:
db.close()
print("\n✅ Notebook 01 complete — Hypothesis Testing, CIs & Effect Sizes")